<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/V2_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [1]:
%pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 26.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
from datetime import datetime, time
import re

from neo4j import GraphDatabase
import math, os

from typing import List, Dict, Any
import pandas as pd
import textwrap
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

from typing import Optional, Iterable, Dict, Any, List
from neo4j import GraphDatabase
import pandas as pd
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

In [3]:
def run_hf_llm(
    df: pd.DataFrame,
    model_path: str,
    prompt_col: str = "prompt_text",
    system_prompt: str = "",
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9,
    messages: list | None = None,   # optional base conversation (few-shot)
    chat_format: str | None = None, # kept for compatibility, unused
    strip_think_tags: bool = True,
) -> pd.DataFrame:
    """
    Generic HF chat-model runner for:
      - HuggingFaceTB/SmolLM3-3B
      - google/gemma-2-9b-it
      - mistralai/Mistral-7B-Instruct-v0.3
      - meta-llama/Llama-3.2-3B-Instruct
    Assumes `df[prompt_col]` contains the row-level user instruction.
    """

    # 1. Load tokenizer + model once
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()

    # 2. Ensure pad token is set
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Copy base messages so we never mutate the caller's list
    base_messages = messages[:] if messages else []

    def render_with_chat_template(msgs: list[dict]) -> str:
        """
        Try to render using the model's chat_template.
        If system role is not supported (e.g. Gemma), fold system_prompt into first user message.
        Fallback to a plain instruct-style prompt if needed.
        """
        # If tokenizer has a native chat template, prefer it
        if hasattr(tokenizer, "apply_chat_template"):
            try:
                return tokenizer.apply_chat_template(
                    msgs,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            except Exception as e:
                err = str(e)

                # Handle: Gemma / others that complain about system role
                if "System role not supported" in err or "system" in err.lower():
                    # Merge ALL system messages into the first user message
                    system_text = "\n\n".join(
                        m["content"] for m in msgs if m.get("role") == "system"
                    ).strip()
                    fixed = []
                    merged = False
                    for m in msgs:
                        role = m.get("role")
                        content = m.get("content", "")
                        if role == "system":
                            continue
                        if role == "user" and system_text and not merged:
                            fixed.append({
                                "role": "user",
                                "content": system_text + "\n\n" + content
                            })
                            merged = True
                        else:
                            fixed.append(m)

                    return tokenizer.apply_chat_template(
                        fixed,
                        tokenize=False,
                        add_generation_prompt=True,
                    )

                # Any other template error → fall through to manual fallback below

        # Fallback: simple instruct-style formatting
        parts = []
        # Inline system_prompt at the top if present
        if system_prompt:
            parts.append(system_prompt.strip())
        # Include any base messages
        for m in base_messages:
            role = m.get("role", "user").upper()
            parts.append(f"{role}:\n{m.get('content', '').strip()}")
        # Last user message is appended by caller
        # Caller will add "USER:\n...ASSISTANT:\n"
        return "\n\n".join(parts)

    def build_prompt_text(user_prompt: str) -> str:
        # Build per-row messages (fresh each time)
        row_messages: list[dict] = []

        # Add system prompt as system role if not already present
        if system_prompt and not any(m.get("role") == "system" for m in base_messages):
            row_messages.append({"role": "system", "content": system_prompt})

        # Add any shared messages (few-shot, context, etc.)
        row_messages.extend(base_messages)

        # Add this row's user content
        row_messages.append({"role": "user", "content": user_prompt})

        rendered = render_with_chat_template(row_messages)

        # If we fell back to simple text above (no ASSISTANT cue yet), add it now
        if "ASSISTANT:" in rendered:
            return rendered
        return rendered + f"\n\nUSER:\n{user_prompt.strip()}\nASSISTANT:\n"

    outputs = []
    time_taken = []
    token_counts = []

    # 3. Loop rows with progress bar
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating"):
        user_prompt = str(row[prompt_col])

        prompt_text = build_prompt_text(user_prompt)
        enc = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        start = time.time()
        with torch.no_grad():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=model.config.pad_token_id,
            )
        end = time.time()

        # Only generated continuation
        gen_only_ids = gen_ids[0][enc["input_ids"].shape[1]:]
        gen_text = tokenizer.decode(gen_only_ids, skip_special_tokens=True).strip()

        if strip_think_tags and gen_text:
            gen_text = (
                gen_text.replace("<think>", "")
                        .replace("</think>", "")
                        .strip()
            )

        token_count = len(tokenizer(gen_text).input_ids) if gen_text else 0

        outputs.append(gen_text)
        time_taken.append(round(end - start, 2))
        token_counts.append(token_count)

    out_df = df.copy()
    out_df["output"] = outputs
    out_df["time_taken"] = time_taken
    out_df["token_count"] = token_counts
    return out_df


In [4]:
# ---------- helpers ----------
def slugify(text: str) -> str:
    """
    Convert a string to lowercase, remove non-alphanumeric characters,
    and replace spaces with underscores.
    """
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)   # keep only letters, numbers, spaces
    text = re.sub(r'\s+', '_', text.strip())  # replace spaces with underscores
    return text

def _split_maybe(s, sep="|"):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    return [t.strip() for t in str(s).split(sep) if t.strip()]

def compress_list(items: List[str], max_len=180) -> str:
    s = ", ".join(dict.fromkeys(t.strip() for t in items if t))
    return (s[:max_len] + "…") if len(s) > max_len else s

def condense_nodes(nodes_df: pd.DataFrame, node_ids: List[str], max_nodes=12) -> str:
    rows = []
    subset = nodes_df.set_index("node_id").reindex(node_ids).dropna(how="all").reset_index()
    subset = subset.assign(
        required=subset["required"].fillna(False).astype(bool),
        phrases=subset["phrases"].apply(_split_maybe)
    )
    # required first
    subset = subset.sort_values(["required","node_id"], ascending=[False, True]).head(max_nodes)
    for _, r in subset.iterrows():
        rows.append(
            f"- {r['node_id']}: {r['name']} — req: {'required' if r['required'] else 'optional'} "
            f"— key_phrases: {compress_list(r['phrases'], 160)}"
        )
    return "\n".join(rows)

def condense_company(ctx: Dict[str, Any]) -> str:
    return textwrap.dedent(f"""\
    - industry: {ctx.get('industry','unknown')}; footprint: {ctx.get('size','?')} / {ctx.get('regions','?')}
    - systems: {compress_list(ctx.get('key_systems', []), 100)}
    - risk posture: {ctx.get('risk_summary','n/a')}
    - constraints: {compress_list(ctx.get('constraints', []), 160)}
    """)

def build_v1_prompt_static(
    policy_title: str,
    section_number: str,
    section_title: str,
    audience: str,
    node_ids: List[str],
    ontology_nodes_df: pd.DataFrame,
    company_ctx: Dict[str, Any],
    hard_limit_tokens: int = 700
) -> str:
    nodes_txt = condense_nodes(ontology_nodes_df, node_ids)
    ctx_txt   = condense_company(company_ctx)
    user_prompt = f"""
[INSTRUCTION]
Draft the section "{section_title}" for the {policy_title} policy. Audience: {audience}.
Focus only on the policy text itself. Tone: professional and directive.

[ONTOLOGY CONTEXT]
{nodes_txt}

[COMPANY CONTEXT]
{ctx_txt}

[FRAMEWORK EXPECTATIONS]
Ensure the section aligns conceptually with the following controls and ontology requirements.
If multiple controls apply, integrate them naturally within the same policy paragraph.

[CONSTRAINTS]
- No prefaces or commentary.
- No explanations or meta text.
- Write as if this text will appear directly in the final policy document.
""".strip()
    return user_prompt

# ---------- batch assembly ----------
def assemble_v1_prompts(
    policy_sections: pd.DataFrame,
    ontology_nodes: pd.DataFrame,
    section_to_nodes: pd.DataFrame,
    company_ctx: Dict[str, Any]
) -> pd.DataFrame:
    # join node_ids
    key = ["policy_title","section_number"]
    nodes = (section_to_nodes.groupby(key)["node_id"]
             .apply(list).reset_index())
    df = policy_sections.merge(nodes, on=key, how="left").fillna({"node_id": []})
    prompts = []
    for _, r in df.iterrows():
        prompt = build_v1_prompt_static(
            policy_title=r["policy_title"],
            section_number=str(r["section_number"]),
            section_title=r["section_title"],
            audience=r.get("audience","General Employees"),
            node_ids=r["node_id"],
            ontology_nodes_df=ontology_nodes,
            company_ctx=company_ctx,
        )
        prompts.append(prompt)
    df["v1_prompt"] = prompts
    return df


In [5]:
def get_section_text(df, policy_id: str, section: str, include_children=True) -> str:
    # Filter for this policy
    pdf = df[df['policy_id'] == policy_id].copy()
    if pdf.empty:
        return f"[No content found for policy {policy_id}]"

    # Choose filtering style: exact match or include children (prefix)
    if include_children:
        section_df = pdf[pdf['clause_section_number'].str.startswith(section)]
    else:
        section_df = pdf[pdf['clause_section_number'] == section]

    if section_df.empty:
        return f"[No section {section} found for policy {policy_id}]"

    # # Sort numerically by section number (handles 5, 5.1, 5.2, 5.10 correctly)
    # section_df = section_df.sort_values(
    #     by="clause_section_number",
    #     key=lambda s: s.str.split('.').apply(lambda x: [int(i) for i in x])
    # )

    output_lines = []
    last_section = None

    for _, row in section_df.iterrows():
        sec_num = row['clause_section_number']
        title = row.get('section_title', '')
        text = str(row.get('clause_text', '')).strip()

        # Print header at first appearance of each section/subsection
        if sec_num != last_section:
            header = f"{sec_num} {title}".strip()
            output_lines.append(header)
            last_section = sec_num

        # Add the body text
        if text:
            output_lines.append(text)

    return "\n".join(output_lines)


In [ ]:
engine = create_engine(connection_url, echo=False)

system_prompt = (
    """
    You are a cybersecurity policy generation assistant.

    Your role is to draft clear, enforceable, framework-aligned policy text by grounding your responses in:

    1. The provided framework ontology
    2. The knowledge graph (KG) mappings that link frameworks → controls → sub-controls → policy → sections
    3. The company context (industry, size, environment, technology stack, regulatory obligations)

    You must **only** use information explicitly provided in the ontology, KG mappings, and context.
    Do NOT invent new requirements, new terminology, or new controls.

    Your writing must be:
    - Professional, concise, directive, and actionable
    - Suitable for inclusion directly inside a formal corporate policy
    - Neutral in tone (no prefaces or meta commentary)
    - Aligned with the user-provided framework expectations

    STRICT PROHIBITIONS:
    - Do NOT include section numbers (e.g., 2.1, 3.4)
    - Do NOT include headings or labels unless explicitly asked
    - Do NOT explain reasoning
    - Do NOT mention frameworks by name inside the text
    - Do NOT reference the ontology or KG
    - Do NOT output analysis, notes, or preambles

    Your output must be ONLY the policy section text.
    """
)


df = pd.read_sql("SELECT * FROM policy_lines where policy_origin = 'client';", engine)
compact_df = df[['policy_id','policy_title','source_framework','policy_origin','section_id','section_title','clause_section_number','company_type','company_id']]
compact_df = compact_df.drop_duplicates()
compact_df['section_text'] = compact_df[['policy_id', 'clause_section_number']].apply(lambda row: get_section_text(df, row['policy_id'], row['clause_section_number']),axis =1)
# compact_df = compact_df.sample(n=100, random_state=42)

In [ ]:
def norm_section(s: str) -> str:
    s = "" if pd.isna(s) else str(s).strip()
    s = re.sub(r"[^0-9.]+", "", s)
    s = re.sub(r"\.+", ".", s).strip(".")
    return s

In [ ]:
CYPHER = """
MATCH (pol:Policy)
WHERE ($policy_id IS NULL OR pol.policy_id = $policy_id)
  AND ($policy_title IS NULL OR toLower(pol.title) = toLower($policy_title))
OPTIONAL MATCH (pol)-[:HAS_UNIT]->(unit:PolicyUnit)
WHERE $section_number IS NULL
   OR toString(unit.number)  = toString($section_number)        // or name (fallback)
OPTIONAL MATCH (unit)-[:HAS_LINE]->(lnA:Line)
OPTIONAL MATCH (pol)-[:HAS_LINE]->(lnB:Line)
// if you actually keep section info on Line, this will still work:
WHERE $section_number IS NULL
   OR toString(lnB.number) = toString($section_number)

WITH pol, unit, coalesce(lnA, lnB) AS ln
WHERE ln IS NOT NULL
MATCH (onto:Ontology)-[r:INFLUENCED_BY]->(ln)
WITH pol, unit, ln, onto, r
ORDER BY coalesce(ln.line_index_in_unit, ln.line_id, 1e9)
RETURN
  pol.policy_id                                        AS policy_id,
  pol.title                                     AS policy_title,
  coalesce(
    ln.number,
    toString($section_number),
    toString(unit.label),
    toString(unit.name)
  )                                             AS section_number,
  ln.line_id                                    AS clause_id,
  coalesce(ln.line_index_in_unit, ln.line_index_in_unit) AS sort_column,
  ln.text                                       AS clause_text,
  onto.node_id                                  AS ontology_id,
  onto.name                                     AS ontology_name,
  onto.description                              AS ontology_description,
  onto.label                                    AS ontology_label,
  onto.seeds                                    AS ontology_seeds,
  onto.counterexamples                          AS ontology_counterexamples,
  r.score                                       AS ontology_relation_score,
  r.method                                      AS ontology_relation_method,

  r.createdAt                                   AS createdAt

"""

def get_related_ontologies(
    driver,
    policy_id: Optional[str] = None,
    policy_title: Optional[str] = None,
    section_number: Optional[str] = None,
) -> pd.DataFrame:
    """
    Fetch Ontology nodes related to clauses for the given Policy/Section.
    Any of the parameters may be None; at least one of policy_id/policy_title is recommended.
    Results are ordered by ln.sort_column if present.
    """
    params = {
        "policy_id": policy_id,
        "policy_title": policy_title,
        "section_number": section_number,
    }
    with driver.session(database=NEO4J_DATABASE) as session:
        recs = session.run(CYPHER, params)
        rows: List[Dict[str, Any]] = [r.data() for r in recs]
    return pd.DataFrame(rows)

# Example usage




In [ ]:
engine = create_engine(connection_url, echo=False)
all_client_ont = pd.read_sql('SELECT * FROM client_ontologies', engine)

In [ ]:
from typing import List

def _listish(x) -> List[str]:
    # Neo4j may return list or None; keep only clean strings
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    if x is None:
        return []
    # defensive: pipe-delimited string
    return [t.strip() for t in str(x).split("|") if t.strip()]

def make_ontology_nodes_df(ont_df_raw: pd.DataFrame, max_phrases_per_node: int = 5) -> pd.DataFrame:
    if ont_df_raw.empty:
        return pd.DataFrame(columns=["node_id","label","phrases"])
    # group by node; prefer ontology_label then name
    grp = (ont_df_raw
           .groupby(["ontology_id", "ontology_label", "ontology_name"], dropna=False)
           .agg({
               "ontology_seeds": lambda s: sorted({p for lst in s for p in _listish(lst)})
           })
           .reset_index())

    grp["node_id"] = grp["ontology_id"]
    grp["label"]   = grp["ontology_label"].fillna(grp["ontology_name"]).fillna(grp["ontology_id"])
    grp["phrases"] = grp["ontology_seeds"].apply(lambda L: L[:max_phrases_per_node])

    out = grp.loc[:, ["node_id","label","phrases"]].copy()
    # stable sort: label then node_id
    out = out.sort_values(["label","node_id"], kind="stable").reset_index(drop=True)
    return out

def render_ontology_block(ontology_nodes_df: pd.DataFrame) -> str:
    if ontology_nodes_df.empty:
        return ""
    lines = []
    for _, r in ontology_nodes_df.iterrows():
        kp = ", ".join(r["phrases"])
        lines.append(f"- {r['node_id']}: {r['label']} — key_phrases: {kp}")
    return "\n".join(lines)

def render_client_context(client_ont_df: pd.DataFrame) -> str:
    # Expect columns like ['client_id','ontology_id','ontology_label','ontology_value']
    if client_ont_df.empty:
        return ""
    lines = []
    for _, r in client_ont_df.iterrows():
        lines.append(f"- {r['ontology_label']}: {r.get('ontology_value', '')}")
    return "\n".join(lines)




In [ ]:
def _is_empty(v) -> bool:
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, (list, tuple, set)):
        return len(v) == 0
    if isinstance(v, str):
        return len(v.strip()) == 0
    return False

def _normalize_value(v) -> str:
    # Turn lists/tuples/sets into comma-separated strings; trim whitespace.
    if isinstance(v, (list, tuple, set)):
        parts = [str(x).strip() for x in v if str(x).strip()]
        return ", ".join(parts)
    return str(v).strip()

def _prettify_label(onto_id: str) -> str:
    # e.g., "organization.tech_stack" -> "Technology Stack"
    last = onto_id.split(".")[-1].replace("_", " ")
    return last.title()

def _shorten(s: str, max_len: int) -> str:
    if max_len is None:
        return s
    return s if len(s) <= max_len else (s[: max_len - 1] + "…")

def client_wide_to_long(
    client_ont_df: pd.DataFrame,
    *,
    allowed_fields: Optional[Iterable[str]] = None,   # whitelist; preserves order
    exclude_fields: Iterable[str] = ("client_id", "organization.name"),
    label_overrides: Optional[Dict[str, str]] = None,
    max_lines: Optional[int] = None,                  # cap number of rows returned
    max_value_len: Optional[int] = 200,               # truncate long values
) -> pd.DataFrame:
    """
    Convert a wide client ontology row into a clean long-form dataframe with:
      - filtering via allowed_fields (whitelist) and exclude_fields
      - normalized values (lists -> comma-separated)
      - friendly label column with optional overrides
      - optional row cap and value shortening

    Returns columns: ['client_id','ontology_id','ontology_value','label']
    """
    label_overrides = label_overrides or {}

    if client_ont_df is None or client_ont_df.empty:
        return pd.DataFrame(columns=["client_id","ontology_id","ontology_value","label"])

    # Melt wide -> long
    long_df = client_ont_df.melt(
        id_vars=["client_id"],
        var_name="ontology_id",
        value_name="ontology_value"
    )

    # Exclude always
    long_df = long_df[~long_df["ontology_id"].isin(set(exclude_fields))]

    # Normalize & drop empties/NaNs
    long_df["ontology_value"] = long_df["ontology_value"].apply(_normalize_value)
    long_df = long_df[~long_df["ontology_value"].apply(_is_empty)].copy()

    # If whitelist provided, filter + preserve whitelist order
    if allowed_fields is not None:
        allowed = [f for f in allowed_fields if f in set(long_df["ontology_id"])]
        long_df = long_df[long_df["ontology_id"].isin(allowed)]
        # preserve whitelist order deterministically
        order_map = {k: i for i, k in enumerate(allowed)}
        long_df["__ord"] = long_df["ontology_id"].map(order_map)
        long_df = long_df.sort_values(["__ord", "ontology_id"], kind="stable").drop(columns="__ord")
    else:
        # deterministic fallback order: by ontology_id
        long_df = long_df.sort_values(["ontology_id"], kind="stable")

    # Labels (pretty + overrides)
    long_df["label"] = long_df["ontology_id"].map(lambda k: label_overrides.get(k, _prettify_label(k)))

    # Shorten values if requested
    if max_value_len is not None:
        long_df["ontology_value"] = long_df["ontology_value"].map(lambda s: _shorten(s, max_value_len))

    # Optional cap on number of lines
    if max_lines is not None:
        long_df = long_df.head(max_lines)

    # Final column order
    return long_df.loc[:, ["client_id","ontology_id","ontology_value","label"]].reset_index(drop=True)

def render_client_context_from_long(long_df: pd.DataFrame) -> str:
    """Convenience: turn the long DF into a bullet list block."""
    if long_df is None or long_df.empty:
        return ""
    return "\n".join(f"- {r['label']}: {r['ontology_value']}" for _, r in long_df.iterrows())


In [ ]:
allowed = []



In [ ]:
# Neo4j Framework Context Extraction
# ============================================================


CYPHER_POLICY_FRAMEWORK_MAP = """
MATCH (f:Framework)
WHERE f.name = $framework_name
MATCH (f)-[:HAS_CONTROL_AREA]->(ca:ControlArea)
MATCH (ca)-[:HAS_SUBCONTROL]->(sc:SubControl)
MATCH (sc)-[:REQUIRES_EVIDENCE]->(er:EvidenceRequirement)
MATCH (p:Policy)-[:SATISFIES_REQUIREMENT]->(er)
WHERE p.title = $policy_name
WITH f, p, ca, sc, er,
     apoc.text.regexGroups(sc.title, ".*\\(([^)]+)\\)") AS groups
RETURN DISTINCT
  f.name              AS framework_name,
  p.title             AS policy_title,
  ca.name             AS control_name,
  ca.code             AS control_id,          // IF you have this
  ca.description      AS control_description, // OR whatever your property is
  sc.title            AS subcontrol_name,
  groups[0][1] AS subcontrol_id,
apoc.text.replace(sc.description, '<[^>]+>', '')   AS subcontrol_description,  // OR sc.text / sc.requirement_text
  er.name            AS evidence_title,
  trim(replace(
    er.example_raw,
    "(document name and content will vary by organization):",
    ""
))     AS evidence_description     // OR er.text / er.requirement
ORDER BY control_name, subcontrol_name


"""

from collections import defaultdict


def get_framework_context_for_policy(policy_name: str,
                                     framework_name: str = "NIST CSF") -> dict:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            rows = session.run(
                CYPHER_POLICY_FRAMEWORK_MAP,
                framework_name=framework_name,
                policy_name=policy_name,
            ).data()
    finally:
        driver.close()

    # Group by control area
    by_control: dict[str, dict] = {}

    for r in rows:
        ctrl_name = r["control_name"]

        if ctrl_name not in by_control:
            by_control[ctrl_name] = {
                "control_area": ctrl_name,
                "control_id": r.get("control_id"),
                "control_description": r.get("control_description"),
                "subcontrols": []
            }

        by_control[ctrl_name]["subcontrols"].append({
            "id": r.get("subcontrol_id"),
            "name": r["subcontrol_name"],
            "description": r.get("subcontrol_description"),
            "evidence_title": r.get("evidence_title"),
            "evidence_description": r.get("evidence_description"),
        })

    return {
        "framework": framework_name,
        "policy": policy_name,
        "controls": list(by_control.values())
    }


In [ ]:
import re

def clean_requirement_text(text: str, max_chars: int = 600) -> str:
    if not text:
        return ""

    # Cut off the "Related Documents" section if present
    parts = re.split(r"(?i)related documents", text, maxsplit=1)
    text = parts[0]

    # Remove "Guidance:" and "Action Items:" prefixes (keep the content)
    text = re.sub(r"(?i)guidance:\s*", "", text)
    text = re.sub(r"(?i)action items:\s*", "", text)

    # Collapse excessive whitespace/newlines
    text = re.sub(r"\s+", " ", text).strip()

    # Truncate to avoid huge blobs
    if len(text) > max_chars:
        # Try not to cut mid-sentence
        truncated = text[:max_chars]
        last_period = truncated.rfind(".")
        if last_period > 100:  # don't cut too aggressively
            truncated = truncated[: last_period + 1]
        text = truncated + " …"

    return text


In [ ]:
def render_framework_context(framework_data: dict) -> str:
    """
    Render a compact, non-duplicative framework context block for prompts.
    """
    lines: list[str] = []

    lines.append(f"Framework: {framework_data['framework']}")
    lines.append(f"Policy: {framework_data['policy']}")
    lines.append("")

    # Track evidence descriptions we've already shown, to avoid repeats
    seen_evidence_desc = set()

    for ctrl in framework_data.get("controls", []):
        cid = ctrl.get("control_id")
        cdesc = ctrl.get("control_description")

        # Control area header
        if cid:
            lines.append(f"Control Area: {ctrl['control_area']} ({cid})")
        else:
            lines.append(f"Control Area: {ctrl['control_area']}")

        if cdesc:
            lines.append(f"  Description: {clean_requirement_text(cdesc, max_chars=300)}")

        for sc in ctrl.get("subcontrols", []):
            sc_id = sc.get("id")
            sc_name = sc.get("name")
            sc_desc_raw = sc.get("description")

            # Sub-control header
            if sc_id:
                lines.append(f"  Sub-control: {sc_id} – {sc_name}")
            else:
                lines.append(f"  Sub-control: {sc_name}")

            # Cleaned requirement (shorter, no related docs)
            sc_req = clean_requirement_text(sc_desc_raw, max_chars=500)
            if sc_req:
                lines.append(f"    Requirement: {sc_req}")

            # Evidence – keep this minimal and de-duplicated
            ev_title = sc.get("evidence_title")
            ev_desc_raw = sc.get("evidence_description")

            if ev_title or ev_desc_raw:
                # Normalize description to dedupe
                ev_desc_norm = (ev_desc_raw or "").strip()
                if ev_desc_norm and ev_desc_norm in seen_evidence_desc:
                    # We've already shown this long boilerplate somewhere else
                    # Just skip it to save tokens
                    continue

                if ev_desc_norm:
                    seen_evidence_desc.add(ev_desc_norm)

                # Option A: just list the title (simple & short)
                if ev_title:
                    lines.append(f"    Evidence Expectation: {ev_title}")

                # Option B (optional): include a single, short line of description
                # first_line = ev_desc_norm.splitlines()[0].strip() if ev_desc_norm else ""
                # if first_line:
                #     lines.append(f"      Note: {clean_requirement_text(first_line, max_chars=200)}")

        lines.append("")  # blank line between control areas

    return "\n".join(lines).strip()


In [ ]:
def add_framework_context_to_df(
    df,
    framework_col: str='source_framework',
    framework_context_col: str='framework_context'
):
    framework_contexts = []
    pbar = tqdm(
        total=len(df),
        desc="Adding framework context",
        unit="clause",
        leave=True
    )

    for index, row in df.iterrows():
        policy_name = row.get('policy_title')
        framework = row.get(framework_col)

        # Update progress bar message dynamically
        pbar.set_postfix({
            "Policy": str(policy_name)[:25],
            "Framework": str(framework)
        })


        framework_data = get_framework_context_for_policy(policy_name, framework)
        context_text = render_framework_context(framework_data)
        framework_contexts.append(context_text)
        pbar.update(1)

    pbar.close()

    df[framework_context_col] = framework_contexts
    return df

In [ ]:
# compact_df = compact_df.sample(n=10, random_state=42)

In [ ]:
compact_df = add_framework_context_to_df(compact_df)

Adding framework context:   0%|          | 0/1297 [00:00<?, ?clause/s]

In [ ]:
def render_kg_mapping_only(framework_data: dict) -> str:
    """
    Very compact view of Framework -> Control Area -> Sub-control IDs/names,
    without the long requirement descriptions.
    """
    lines = []
    lines.append(f"Framework: {framework_data['framework']}")
    lines.append(f"Policy: {framework_data['policy']}")
    lines.append("")

    for ctrl in framework_data.get("controls", []):
        cid = ctrl.get("control_id")
        if cid:
            lines.append(f"Control Area: {ctrl['control_area']} ({cid})")
        else:
            lines.append(f"Control Area: {ctrl['control_area']}")

        for sc in ctrl.get("subcontrols", []):
            sc_id = sc.get("id")
            sc_name = sc.get("name")
            if sc_id:
                lines.append(f"  - Sub-control: {sc_id} – {sc_name}")
            else:
                lines.append(f"  - Sub-control: {sc_name}")

        lines.append("")

    return "\n".join(lines).strip()


In [ ]:
def gen_v2_prompt(policy):
    policy_id      = policy.get("policy_id")
    policy_title   = policy.get("policy_title")
    section_number = policy.get("clause_section_number")
    client_id      = policy.get("company_id")
    section_title  = policy.get("section_title")
    framework      = policy.get("source_framework") or "NIST CSF"

    # Optional: audience from row, with sensible default
    audience = policy.get(
        "audience",
        "all employees, contractors, and relevant third parties"
    )

    # 1) Ontology slice for this policy + section
    ont_df_raw = get_related_ontologies(
        driver,
        policy_id=policy_id,
        policy_title=policy_title,
        section_number=section_number,
    )

    # 2) Collapse to prompt-ready ontology nodes
    ontology_nodes_df = make_ontology_nodes_df(
        ont_df_raw,
        max_phrases_per_node=5,
    )

    # 3) Client ontology slice (deterministic org context)
    client_ont = all_client_ont[
        all_client_ont["client_id"] == client_id
    ].copy()

    # 4) Build ontology context text block for the prompt
    ontology_text = render_ontology_block(ontology_nodes_df)

    # 5) Build client/company context block
    long_ctx = client_wide_to_long(
        client_ont,
        allowed_fields=allowed,
        exclude_fields=("client_id", "organization.name"),
        label_overrides={"organization.tech_stack": "Technology Stack"},
        max_lines=8,
        max_value_len=180,
    )
    company_context = render_client_context_from_long(long_ctx)

    # 6) Framework / KG context for this policy
    framework_data = get_framework_context_for_policy(
        policy_name=policy_title,
        framework_name=framework,
    )

    # KG-only mapping (IDs/names) vs detailed expectations (requirements)
    kg_text       = render_kg_mapping_only(framework_data)
    controls_text = render_framework_context(framework_data)

    # 7) Assemble the final V2 user prompt
    user_prompt = f"""
[INSTRUCTION]
Draft the policy section titled: "{section_title}"
for the overall policy: "{policy_title}".
Audience: {audience}.

Focus exclusively on producing the final policy text itself (no headings, no prefaces).

[ONTOLOGY CONTEXT]
Below is the ontology for relevant cybersecurity concepts tied to this policy.
Use the definitions, relationships, required behaviors, and expectations exactly as provided.
Do not create new ontology items.

{ontology_text}

[KG MAPPINGS]
These are the knowledge graph (KG) relationships that connect:
Framework → Control Area → Sub-control → Policy → Section.

Use these relationships to ensure your text aligns precisely with the mapped framework expectations.

{kg_text}

[FRAMEWORK EXPECTATIONS]
Incorporate the mapped control requirements below into the policy text.
If multiple controls apply, integrate them naturally into one coherent section.
Do not mention the control IDs or frameworks explicitly.

{controls_text}

[COMPANY CONTEXT]
Use only the information relevant to operationalizing the policy within this organization’s environment.
Do not restate this context verbatim inside the policy.

{company_context}

[CONSTRAINTS]
- No prefaces, explanations, or meta-text.
- No policy numbers or section labels.
- No references to frameworks, controls, or ontology.
- Write as if the final output will be pasted directly into the formal policy.
- Ground all content strictly in the ontology + KG mappings + company context, not general knowledge.
- If few or no controls apply (e.g., Purpose, Scope), write a simple, high-quality section appropriate for that title.

Now write the policy section.

<META>section_type={slugify(section_title)}; policy_type={slugify(policy_title)}; framework={slugify(framework)}</META>
""".strip()

    return user_prompt


In [ ]:
compact_df['v2_prompt'] = compact_df.apply(lambda row: gen_v2_prompt(row), axis=1)

In [ ]:
# compact_df = compact_df.sample(n=5, random_state=42)

In [ ]:
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm
model_name = "Gemma-2-9B"
model_path =  "google/gemma-2-9b-it"
messages = []
chat_format= "gemma-2"

out_df = run_hf_llm(
    compact_df.rename(columns={"v2_prompt": "v2_prompt"}),  # just clarity
    model_path=model_path,
    max_new_tokens=256,
    temperature=0.2,
    top_p=0.9,
    prompt_col="v2_prompt",
    messages=messages,
    system_prompt=system_prompt,
    chat_format=chat_format,
    strip_think_tags=True,

)



`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating:   0%|          | 0/1297 [00:00<?, ?it/s]

In [ ]:
df = compact_df.copy()
out_df_m= out_df.copy()
df.rename(columns={"v2_prompt": "prompt_text"}, inplace=True)

# Ensure consistent types
df["policy_id"] = df["policy_id"].astype(str)

# Unique prompt_text -> all policy_ids that use it
prompt_map = (
    df.groupby("prompt_text", as_index=False)
      .agg(policy_ids=("policy_id", lambda s: sorted(s.unique())))
)

out_df_m.rename(columns={"v2_prompt": "prompt_text"}, inplace=True)
if "policy_id" in out_df_m.columns:
    out_df_m = out_df_m.drop(columns=["policy_id"])

model_mapping = (
    out_df_m.merge(prompt_map, on="prompt_text", how="left")
          .explode("policy_ids")
          .rename(columns={"policy_ids": "policy_id"})
)


final = model_mapping[['policy_id','output','time_taken','token_count','policy_title','clause_section_number']]
final = final.copy()
final.rename(columns={'clause_section_number': 'section_number'}, inplace=True)
final['model_name'] = model_path.split('/')[1]
final['execution_date'] = datetime.now()
final['model_type'] = 'V2'
final['variant'] = 'A'
# "policy_id"	"output"	"time_taken"	"token_count"	"policy_title"	"section_number"
# "model_name"	"execution_date" "model_type"
# "variant"



In [ ]:
final = final[["policy_id", "model_name", "output", "time_taken", "token_count", "policy_title", "section_number", "variant", "model_type", "execution_date"]]


In [ ]:
final

,policy_id,model_name,output,time_taken,token_count,policy_title,section_number,variant,model_type,execution_date
0,CL_0001-AUP-v2,gemma-2-9b-it,This policy outlines acceptable use of company...,2.94,33,Acceptable Use Policy,1,A,V2,2025-11-19 01:11:23.880515
0,CL_0006-AUP-v1.0,gemma-2-9b-it,This policy outlines acceptable use of company...,2.94,33,Acceptable Use Policy,1,A,V2,2025-11-19 01:11:23.880515
0,CL_0007-AUP-v1.0,gemma-2-9b-it,This policy outlines acceptable use of company...,2.94,33,Acceptable Use Policy,1,A,V2,2025-11-19 01:11:23.880515
1,CL_0001-AUP-v2,gemma-2-9b-it,"This policy applies to all employees, contract...",1.81,26,Acceptable Use Policy,2,A,V2,2025-11-19 01:11:23.880515
1,CL_0006-AUP-v1.0,gemma-2-9b-it,"This policy applies to all employees, contract...",1.81,26,Acceptable Use Policy,2,A,V2,2025-11-19 01:11:23.880515
...,...,...,...,...,...,...,...,...,...,...
1292,CL_0007-RAM-v1.0,gemma-2-9b-it,"When transferring risk to a third party, the o...",2.22,31,Risk Assessment Methodology,3.5.3,A,V2,2025-11-19 01:11:23.880515
1293,CL_0007-RAM-v1.0,gemma-2-9b-it,Risk assessments will be conducted on an annua...,0.91,11,Risk Assessment Methodology,3.6,A,V2,2025-11-19 01:11:23.880515
1294,CL_0007-RAM-v1.0,gemma-2-9b-it,"All employees, contractors, and relevant third...",3.61,54,Risk Assessment Methodology,3.7,A,V2,2025-11-19 01:11:23.880515
1295,CL_0007-RAM-v1.0,gemma-2-9b-it,All identified risks will be documented and ev...,4.89,73,Risk Assessment Methodology,3.8,A,V2,2025-11-19 01:11:23.880515


In [ ]:
final.to_sql('model_results2', engine, if_exists='append', index=False)

433